# Bước 5: Khai thác luật kết hợp
## Đề 13: Phân tích hội thoại trong DVKH trực tuyến
**Mục tiêu:** Tìm cặp câu hỏi–trả lời thường xuất hiện cùng nhau

In [ ]:
import pandas as pd
import numpy as np
from collections import Counter
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

df = pd.read_csv('../data/final/classified_data.csv')
print(f'Loaded: {len(df)} dòng, {df["conv_id"].nunique()} đoạn')

## 5.1 Tạo giao dịch (Cặp hỏi–đáp)

In [ ]:
# Stopwords
stopwords_vi = {'của', 'và', 'là', 'có', 'được', 'cho', 'không', 'này', 'đã',
                'với', 'các', 'trong', 'những', 'nhưng', 'một', 'người', 'để',
                'từ', 'hay', 'tôi', 'mình', 'bạn', 'shop', 'ơi', 'ạ', 'nhé',
                'dạ', 'thì', 'mà', 'cái', 'đó', 'rồi', 'nha', 'luôn', 'quá',
                'lắm', 'vậy', 'thôi', 'đi', 'giúp', 'cho', 'anh', 'chị', 'em',
                'bị', 'lại', 'cũng', 'như', 'nào', 'còn', 'khi', 'sao', 'the',
                'về', 'đây', 'xin', 'lỗi', 'biết', 'phải', 'đang', 'sẽ', 'nên',
                'hay', 'tại', 'theo', 'trên', 'dưới', 'hoặc', 'nếu'}

def extract_keywords(text, n=5):
    """Trích xuất top keywords từ text"""
    if pd.isna(text):
        return []
    words = str(text).lower().split()
    words = [w for w in words if w not in stopwords_vi and len(w) > 1]
    # Lấy top n từ phổ biến nhất (hoặc tất cả nếu ít hơn)
    freq = Counter(words)
    return [w for w, _ in freq.most_common(n)]


# Tạo giao dịch: mỗi đoạn hội thoại = 1 giao dịch
# Items = keywords từ cả customer và agent
transactions = []

for conv_id in df['conv_id'].unique():
    conv = df[df['conv_id'] == conv_id]
    
    customer_text = ' '.join(conv[conv['speaker'] == 'customer']['message_clean'].dropna().astype(str))
    agent_text = ' '.join(conv[conv['speaker'] == 'agent']['message_clean'].dropna().astype(str))
    
    kw_customer = ['KH_' + w for w in extract_keywords(customer_text, 4)]
    kw_agent = ['TV_' + w for w in extract_keywords(agent_text, 4)]
    
    items = kw_customer + kw_agent
    if len(items) >= 2:
        transactions.append(items)

print(f'Tổng giao dịch: {len(transactions)}')
print(f'Ví dụ 3 giao dịch đầu:')
for i, t in enumerate(transactions[:3]):
    print(f'  [{i+1}] {t}')

## 5.2 Chạy Apriori

In [ ]:
try:
    from mlxtend.preprocessing import TransactionEncoder
    from mlxtend.frequent_patterns import apriori, association_rules
    
    # Encode transactions
    te = TransactionEncoder()
    te_array = te.fit_transform(transactions)
    basket_df = pd.DataFrame(te_array, columns=te.columns_)
    print(f'Basket shape: {basket_df.shape}')
    
    # Apriori
    frequent_items = apriori(basket_df, min_support=0.02, use_colnames=True)
    print(f'Frequent itemsets: {len(frequent_items)}')
    
    # Association Rules
    rules = association_rules(frequent_items, metric='confidence', min_threshold=0.3)
    rules = rules.sort_values('lift', ascending=False)
    print(f'Luật kết hợp: {len(rules)}')
    
    print('\nTop 15 luật kết hợp (theo lift):')
    for _, rule in rules.head(15).iterrows():
        ant = ', '.join(list(rule['antecedents']))
        con = ', '.join(list(rule['consequents']))
        print(f'  {ant} → {con}')
        print(f'    support={rule["support"]:.3f}, confidence={rule["confidence"]:.3f}, lift={rule["lift"]:.3f}')

except ImportError:
    print('Cần cài: pip install mlxtend')
    rules = pd.DataFrame()

## 5.3 Lọc luật có ý nghĩa (KH → TV)

In [ ]:
if len(rules) > 0:
    # Lọc luật KH → TV (khách hàng hỏi → tư vấn viên trả lời)
    meaningful_rules = []
    for _, rule in rules.iterrows():
        ant = list(rule['antecedents'])
        con = list(rule['consequents'])
        has_kh = any(a.startswith('KH_') for a in ant)
        has_tv = any(c.startswith('TV_') for c in con)
        if has_kh and has_tv and rule['lift'] > 1.0:
            meaningful_rules.append(rule)
    
    meaningful_df = pd.DataFrame(meaningful_rules)
    print(f'Luật có ý nghĩa (KH→TV, lift>1): {len(meaningful_df)}')
    
    if len(meaningful_df) > 0:
        print('\nTop 10 luật KH→TV:')
        for _, rule in meaningful_df.head(10).iterrows():
            ant = ', '.join([a.replace('KH_','') for a in rule['antecedents']])
            con = ', '.join([c.replace('TV_','') for c in rule['consequents']])
            print(f'  KH nói [{ant}] → TV nên trả lời [{con}]')
            print(f'    confidence={rule["confidence"]:.3f}, lift={rule["lift"]:.3f}')
else:
    meaningful_df = pd.DataFrame()

## 5.4 Trực quan hóa

In [ ]:
if len(rules) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    
    # Scatter: Support vs Confidence
    scatter = axes[0].scatter(rules['support'], rules['confidence'],
                             c=rules['lift'], cmap='RdYlGn', s=50, alpha=0.7, edgecolors='black')
    axes[0].set_xlabel('Support')
    axes[0].set_ylabel('Confidence')
    axes[0].set_title('Support vs Confidence (color=Lift)')
    plt.colorbar(scatter, ax=axes[0], label='Lift')
    
    # Bar: Top 15 rules by lift
    top_rules = rules.head(15).copy()
    top_rules['rule_str'] = top_rules.apply(
        lambda r: ' → '.join([', '.join(list(r['antecedents'])), ', '.join(list(r['consequents']))]),
        axis=1
    )
    # Truncate long strings
    top_rules['rule_short'] = top_rules['rule_str'].str[:50]
    
    axes[1].barh(range(len(top_rules)), top_rules['lift'].values,
                 color='steelblue', edgecolor='black')
    axes[1].set_yticks(range(len(top_rules)))
    axes[1].set_yticklabels(top_rules['rule_short'].values, fontsize=8)
    axes[1].invert_yaxis()
    axes[1].set_xlabel('Lift')
    axes[1].set_title('Top 15 luật kết hợp (theo Lift)')
    
    plt.tight_layout()
    plt.savefig('../results/figures/04_association/support_confidence_scatter.png', dpi=150, bbox_inches='tight')
    plt.show()

## 5.5 Lưu kết quả

In [ ]:
if len(rules) > 0:
    # Chuyển frozenset thành string để lưu CSV
    save_rules = rules.head(30).copy()
    save_rules['antecedents'] = save_rules['antecedents'].apply(lambda x: ', '.join(list(x)))
    save_rules['consequents'] = save_rules['consequents'].apply(lambda x: ', '.join(list(x)))
    save_rules = save_rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]
    
    save_rules.to_csv('../data/final/association_rules.csv', index=False, encoding='utf-8-sig')
    print('✅ Saved: association_rules.csv')
    
    save_rules.to_csv('../results/tables/top_30_rules.csv', index=False, encoding='utf-8-sig')
    print('✅ Saved: top_30_rules.csv')
    
    print(f'\nTổng kết: {len(rules)} luật, {len(meaningful_df)} luật KH→TV có ý nghĩa')
else:
    print('Không có luật kết hợp. Thử giảm min_support.')